In [4]:
# CM3015 Breast Cancer Detection
# Notebook 07: Optuna Hyperparameter Tuning + ConvNeXt-Nano Manual Fix Retrain

import os
import json
import time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import timm
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

from sklearn.metrics import (
    f1_score, recall_score, roc_auc_score,
    roc_curve, confusion_matrix
)
from sklearn.calibration import calibration_curve

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
print(f"timm    : {timm.__version__}")
print(f"optuna  : {optuna.__version__}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

PyTorch : 2.10.0+cu128
CUDA    : True
timm    : 1.0.26
optuna  : 4.9.0
GPU     : Tesla T4


Cell 2 — Paths and control flags

In [5]:
NB02 = Path("/kaggle/input/notebooks/mfjmrizvi/02-mil-patch-extraction")
NB03 = Path("/kaggle/input/notebooks/mfjmrizvi/03-efficientnet-b0")
NB05 = Path("/kaggle/input/notebooks/mfjmrizvi/05-swint")
OUT  = Path("/kaggle/working")

X_TRAIN_PATH       = NB02 / "X_train_patches.npy"
Y_TRAIN_PATH       = NB02 / "y_train_labels.npy"
BAG_IDS_TRAIN_PATH = NB02 / "bag_ids_train.npy"
TRAIN_BAG_IDX_PATH = NB02 / "train_bag_indices.npy"
VAL_BAG_IDX_PATH   = NB02 / "val_bag_indices.npy"
CLASS_WEIGHTS_PATH = NB02 / "class_weights.json"
X_TEST_PATH        = NB02 / "X_test_patches.npy"
Y_TEST_PATH        = NB02 / "y_test_labels.npy"
BAG_IDS_TEST_PATH  = NB02 / "bag_ids_test.npy"

EFFNET_S1_WEIGHTS = NB03 / "efficientnet_b0_stage1.pth"
SWINT_S1_WEIGHTS  = NB05 / "swin_t_stage1.pth"

# ── Control flags ──
N_TRIALS_EFFNET   = 25   # reduce to 15 if GPU budget tight (Mid-Term 3.9.3)
N_TRIALS_SWINT    = 25
RUN_CONVNEXT_OPTUNA = False  # only flip True if budget allows after above two

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
_mean_gpu = torch.tensor(IMAGENET_MEAN, dtype=torch.float32, device=DEVICE).view(1,3,1,1)
_std_gpu  = torch.tensor(IMAGENET_STD,  dtype=torch.float32, device=DEVICE).view(1,3,1,1)

Device: cuda


Cell 3 — Load NB02 data (identical to NB03/04/05)

In [6]:
X_train_all       = np.load(X_TRAIN_PATH)
y_train_all       = np.load(Y_TRAIN_PATH)
bag_ids_all       = np.load(BAG_IDS_TRAIN_PATH)
train_bag_indices = np.load(TRAIN_BAG_IDX_PATH)
val_bag_indices   = np.load(VAL_BAG_IDX_PATH)
X_test_all        = np.load(X_TEST_PATH)
y_test_all        = np.load(Y_TEST_PATH)
bag_ids_test      = np.load(BAG_IDS_TEST_PATH)

with open(CLASS_WEIGHTS_PATH) as f:
    raw_cw = json.load(f)
class_weight_dict = {int(k): float(v) for k, v in raw_cw.items()}

train_mask = np.isin(bag_ids_all, train_bag_indices)
val_mask   = np.isin(bag_ids_all, val_bag_indices)

X_tr, y_tr, bag_tr   = X_train_all[train_mask], y_train_all[train_mask], bag_ids_all[train_mask]
X_val, y_val, bag_val = X_train_all[val_mask],  y_train_all[val_mask],  bag_ids_all[val_mask]

print(f"Train patches: {X_tr.shape}  Val patches: {X_val.shape}")
print(f"Class weights: {class_weight_dict}")

Train patches: (16198, 224, 224, 1)  Val patches: (3572, 224, 224, 1)
Class weights: {0: 0.7259841363102233, 1: 1.6062723431914203}


Cell 4 — Shared model classes (identical to NB03/04/05, dropout added to BagClassifier for tuning)

In [9]:
class PatchClassifier(nn.Module):
    def __init__(self, backbone, feat_dim):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(feat_dim, 1)
    def forward(self, x):
        return self.head(self.backbone(x)).squeeze(1)


class AttentionPool(nn.Module):
    def __init__(self, feat_dim, attn_dim, dropout=0.0):
        super().__init__()
        self.V = nn.Linear(feat_dim, attn_dim, bias=True)
        self.w = nn.Linear(attn_dim, 1, bias=False)
        self.dropout = nn.Dropout(dropout)
    def forward(self, h):
        a = torch.softmax(self.w(self.dropout(torch.tanh(self.V(h)))), dim=0)
        z = torch.sum(a * h, dim=0, keepdim=True)
        return z, a


class BagClassifier(nn.Module):
    # NOTE: dropout param added here — not present in NB03/04/05 baseline models.
    # This is the Optuna-tunable regularisation knob for Stage 2. Flag in report
    # as a structural addition specific to NB07 tuning, distinct from the
    # untuned baseline architecture in NB03-05.
    def __init__(self, feat_dim, attn_dim, dropout=0.0):
        super().__init__()
        self.attn = AttentionPool(feat_dim, attn_dim, dropout)
        self.head = nn.Linear(feat_dim, 1)
    def forward(self, h):
        z, a = self.attn(h)
        return self.head(z).squeeze(), a


class CachedBagDataset(Dataset):
    def __init__(self, features, y_np, bag_ids, bag_list):
        self.features, self.y, self.bag_ids, self.bag_list = features, y_np, bag_ids, bag_list
    def __len__(self):
        return len(self.bag_list)
    def __getitem__(self, idx):
        bid = self.bag_list[idx]
        mask = self.bag_ids == bid
        h = torch.from_numpy(self.features[mask]).float()
        bag_label = torch.tensor(self.y[mask][0], dtype=torch.float32)
        return h, bag_label

def collate_cached(batch):
    return [item[0] for item in batch], torch.stack([item[1] for item in batch])


def build_backbone(model_name, pretrained=True):
    return timm.create_model(model_name, pretrained=pretrained, num_classes=0)


def compute_all_metrics(y_true, y_probs, threshold, label=""):
    y_pred = (y_probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    fpr_val     = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    prob_true, prob_pred = calibration_curve(y_true, y_probs, n_bins=10, strategy='uniform')
    bin_counts = np.histogram(y_probs, bins=np.linspace(0,1,11))[0]
    ece = float(np.sum(bin_counts[:len(prob_true)] * np.abs(prob_true - prob_pred)) / len(y_probs))
    metrics = {
        "threshold": threshold,
        "auc": float(roc_auc_score(y_true, y_probs)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "fpr": float(fpr_val),
        "ece": ece,
    }
    if label:
        print(f"── {label} ──")
        for k, v in metrics.items():
            print(f"  {k:15s}: {v:.4f}")
    return metrics

Part A — ConvNeXt-Nano manual hyperparameter fix retrain (ISS-001)
Cell 5 — Stage 1 retrain with corrected hyperparameters

In [10]:
MODEL_NAME  = "convnext_nano"
ATTN_DIM    = 128
PATCH_SIZE  = 224

BS_STAGE1   = 128        # unchanged — held constant for cross-comparison
LR_STAGE1   = 1e-4       # was 1e-3
EPOCHS_S1   = 25         # was 15
PATIENCE_S1 = 7          # was 5
LR_S2_HEAD  = 5e-5       # was 1e-4 — fixed, not tuned (manual fix only)

S1_WEIGHTS_V2   = OUT / "convnext_nano_v2_stage1.pth"
S2_WEIGHTS_V2   = OUT / "convnext_nano_v2_stage2.pth"
RESULTS_JSON_V2 = OUT / "convnext_nano_v2_results.json"

backbone = build_backbone(MODEL_NAME, pretrained=True).to(DEVICE)
with torch.no_grad():
    _dummy = torch.zeros(2, 3, PATCH_SIZE, PATCH_SIZE, device=DEVICE)
    FEAT_DIM = backbone(_dummy).shape[1]
del _dummy
print(f"FEAT_DIM confirmed: {FEAT_DIM}")

class PatchDataset(Dataset):
    def __init__(self, X_np, y_np):
        self.X = X_np
        self.y = torch.from_numpy(y_np).float()
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        patch = torch.from_numpy(self.X[idx]).float()
        return patch.permute(2, 0, 1), self.y[idx]

patch_model = PatchClassifier(backbone, FEAT_DIM).to(DEVICE)
pos_weight_val = class_weight_dict[1] / class_weight_dict[0]
criterion_s1 = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_val, device=DEVICE))
optimiser_s1 = optim.Adam(patch_model.parameters(), lr=LR_STAGE1)
scheduler_s1 = optim.lr_scheduler.ReduceLROnPlateau(optimiser_s1, mode='min', factor=0.5, patience=3)

train_patch_dl = DataLoader(PatchDataset(X_tr, y_tr), batch_size=BS_STAGE1, shuffle=True, num_workers=2, pin_memory=True)
val_patch_dl   = DataLoader(PatchDataset(X_val, y_val), batch_size=BS_STAGE1, shuffle=False, num_workers=2, pin_memory=True)

scaler = torch.amp.GradScaler('cuda')
s1_history = {"train_loss": [], "val_loss": [], "val_auc": [], "val_f1": []}
best_val_loss_s1, patience_counter_s1 = float("inf"), 0
t0 = time.time()

for epoch in range(1, EPOCHS_S1 + 1):
    patch_model.train()
    running_loss = 0.0
    for patches, labels in train_patch_dl:
        patches, labels = patches.to(DEVICE), labels.to(DEVICE)
        patches = patches.repeat(1, 3, 1, 1)
        patches = (patches - _mean_gpu) / _std_gpu
        optimiser_s1.zero_grad()
        with torch.amp.autocast('cuda'):
            logits = patch_model(patches)
            loss = criterion_s1(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimiser_s1)
        scaler.update()
        running_loss += loss.item() * len(labels)
    train_loss = running_loss / len(train_patch_dl.dataset)

    patch_model.eval()
    val_loss, all_probs, all_labels = 0.0, [], []
    with torch.no_grad():
        for patches, labels in val_patch_dl:
            patches, labels = patches.to(DEVICE), labels.to(DEVICE)
            patches = patches.repeat(1, 3, 1, 1)
            patches = (patches - _mean_gpu) / _std_gpu
            with torch.amp.autocast('cuda'):
                logits = patch_model(patches)
                loss_val = criterion_s1(logits, labels)
            val_loss += loss_val.item() * len(labels)
            all_probs.extend(torch.sigmoid(logits).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    val_loss /= len(val_patch_dl.dataset)
    all_probs, all_labels = np.array(all_probs), np.array(all_labels)
    val_preds = (all_probs >= 0.5).astype(int)
    val_auc = roc_auc_score(all_labels, all_probs)
    val_f1  = f1_score(all_labels, val_preds, zero_division=0)

    s1_history["train_loss"].append(train_loss)
    s1_history["val_loss"].append(val_loss)
    s1_history["val_auc"].append(val_auc)
    s1_history["val_f1"].append(val_f1)
    scheduler_s1.step(val_loss)

    print(f"Ep {epoch:02d}/{EPOCHS_S1}  train={train_loss:.4f}  val={val_loss:.4f}  auc={val_auc:.4f}  f1={val_f1:.4f}")

    if val_loss < best_val_loss_s1:
        best_val_loss_s1, patience_counter_s1 = val_loss, 0
        torch.save(patch_model.state_dict(), S1_WEIGHTS_V2)
        print("  ✓ Saved best Stage 1 weights (v2)")
    else:
        patience_counter_s1 += 1
        if patience_counter_s1 >= PATIENCE_S1:
            print(f"  Early stopping at epoch {epoch}")
            break

print(f"\nStage 1 (v2) complete — {(time.time()-t0)/60:.1f} min")

model.safetensors:   0%|          | 0.00/62.4M [00:00<?, ?B/s]

FEAT_DIM confirmed: 640
Ep 01/25  train=0.7602  val=0.7609  auc=0.7505  f1=0.6282
  ✓ Saved best Stage 1 weights (v2)
Ep 02/25  train=0.6546  val=0.7284  auc=0.7983  f1=0.6657
  ✓ Saved best Stage 1 weights (v2)
Ep 03/25  train=0.6279  val=0.7268  auc=0.7951  f1=0.6509
  ✓ Saved best Stage 1 weights (v2)
Ep 04/25  train=0.6097  val=0.7110  auc=0.7971  f1=0.6756
  ✓ Saved best Stage 1 weights (v2)
Ep 05/25  train=0.5872  val=0.7551  auc=0.7913  f1=0.6453
Ep 06/25  train=0.5744  val=0.7664  auc=0.7884  f1=0.6710
Ep 07/25  train=0.5450  val=0.7502  auc=0.7840  f1=0.6549
Ep 08/25  train=0.5052  val=0.8966  auc=0.7903  f1=0.6632
Ep 09/25  train=0.3992  val=0.8640  auc=0.7781  f1=0.6323
Ep 10/25  train=0.3335  val=1.1442  auc=0.7754  f1=0.6317
Ep 11/25  train=0.2507  val=1.1697  auc=0.7636  f1=0.6474
  Early stopping at epoch 11

Stage 1 (v2) complete — 11.1 min


Cell 6 — Stage 1 (v2) patch-level diagnostics — compare against ISS-001 baseline (patch AUC 0.7250)

In [13]:
patch_model.load_state_dict(torch.load(S1_WEIGHTS_V2, map_location=DEVICE))
patch_model.eval()

all_probs, all_labels = [], []
with torch.no_grad():
    for patches, labels in val_patch_dl:
        patches = patches.to(DEVICE)
        patches = patches.repeat(1, 3, 1, 1)
        patches = (patches - _mean_gpu) / _std_gpu
        with torch.amp.autocast('cuda'):
            all_probs.extend(torch.sigmoid(patch_model(patches)).cpu().numpy())
        all_labels.extend(labels.numpy())

all_probs, all_labels = np.array(all_probs), np.array(all_labels)
preds = (all_probs >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(all_labels, preds).ravel()

s1_metrics_v2 = {
    "patch_auc": float(roc_auc_score(all_labels, all_probs)),
    "patch_f1": float(f1_score(all_labels, preds, zero_division=0)),
    "patch_sensitivity": float(recall_score(all_labels, preds, zero_division=0)),
    "patch_specificity": float(tn / (tn + fp)),
}
print("── ConvNeXt-Nano Stage 1 (v2) Patch-Level Val Metrics ──")
for k, v in s1_metrics_v2.items():
    print(f"  {k:25s}: {v:.4f}")
print(f"\n  Baseline (NB04) patch_auc was 0.7250 — compare against this to confirm ISS-001 fix worked")

── ConvNeXt-Nano Stage 1 (v2) Patch-Level Val Metrics ──
  patch_auc                : 0.7971
  patch_f1                 : 0.6756
  patch_sensitivity        : 0.5953
  patch_specificity        : 0.9031

  Baseline (NB04) patch_auc was 0.7250 — compare against this to confirm ISS-001 fix worked


Cell 7 — Feature extraction (cached to disk this time, for later reuse if needed)

In [14]:
feature_extractor = patch_model.backbone
feature_extractor.eval()
for p in feature_extractor.parameters():
    p.requires_grad_(False)

def extract_features(X_np, model, batch_size=256):
    all_feats = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(X_np), batch_size):
            batch_np = X_np[start:start+batch_size]
            t = torch.from_numpy(batch_np).float().permute(0, 3, 1, 2).to(DEVICE)
            t = t.repeat(1, 3, 1, 1)
            t = (t - _mean_gpu) / _std_gpu
            with torch.amp.autocast('cuda'):
                feats = model(t).cpu().numpy()
            all_feats.append(feats)
    return np.concatenate(all_feats, axis=0)

print("Extracting ConvNeXt-Nano (v2) features...")
feats_tr_cnx   = extract_features(X_tr, feature_extractor)
feats_val_cnx  = extract_features(X_val, feature_extractor)
feats_test_cnx = extract_features(X_test_all, feature_extractor)
print(f"  train : {feats_tr_cnx.shape}  val : {feats_val_cnx.shape}  test : {feats_test_cnx.shape}")

np.save(OUT / "convnext_nano_v2_feats_tr.npy", feats_tr_cnx)
np.save(OUT / "convnext_nano_v2_feats_val.npy", feats_val_cnx)
np.save(OUT / "convnext_nano_v2_feats_test.npy", feats_test_cnx)

Extracting ConvNeXt-Nano (v2) features...
  train : (16198, 640)  val : (3572, 640)  test : (5997, 640)


Cell 8 — Stage 2 training with fixed manual-fix hyperparameters (no Optuna)

In [15]:
bag_model = BagClassifier(FEAT_DIM, ATTN_DIM, dropout=0.0).to(DEVICE)
criterion_s2 = nn.BCEWithLogitsLoss()
optimiser_s2 = optim.Adam(bag_model.parameters(), lr=LR_S2_HEAD)
scheduler_s2 = optim.lr_scheduler.ReduceLROnPlateau(optimiser_s2, mode='min', factor=0.5, patience=3)

EPOCHS_S2, PATIENCE_S2 = 30, 7

train_bag_ds = CachedBagDataset(feats_tr_cnx, y_tr, bag_tr, train_bag_indices)
val_bag_ds   = CachedBagDataset(feats_val_cnx, y_val, bag_val, val_bag_indices)
train_bag_dl = DataLoader(train_bag_ds, batch_size=1, shuffle=True, collate_fn=collate_cached)
val_bag_dl   = DataLoader(val_bag_ds, batch_size=1, shuffle=False, collate_fn=collate_cached)

s2_history = {"train_loss": [], "val_loss": [], "val_auc": [], "val_f1": []}
best_val_loss_s2, patience_counter_s2 = float("inf"), 0
t0 = time.time()

for epoch in range(1, EPOCHS_S2 + 1):
    bag_model.train()
    running_loss = 0.0
    for h_list, labels in train_bag_dl:
        h, label = h_list[0].to(DEVICE), labels[0].to(DEVICE)
        optimiser_s2.zero_grad()
        logit, _ = bag_model(h)
        loss = criterion_s2(logit, label)
        loss.backward()
        optimiser_s2.step()
        running_loss += loss.item()
    train_loss = running_loss / len(train_bag_ds)

    bag_model.eval()
    val_loss, val_probs, val_true = 0.0, [], []
    with torch.no_grad():
        for h_list, labels in val_bag_dl:
            h, label = h_list[0].to(DEVICE), labels[0].to(DEVICE)
            logit, _ = bag_model(h)
            val_loss += criterion_s2(logit, label).item()
            val_probs.append(torch.sigmoid(logit).item())
            val_true.append(label.item())
    val_loss /= len(val_bag_ds)
    val_probs, val_true = np.array(val_probs), np.array(val_true)
    val_preds = (val_probs >= 0.5).astype(int)
    val_auc = roc_auc_score(val_true, val_probs)
    val_f1  = f1_score(val_true, val_preds, zero_division=0)

    s2_history["train_loss"].append(train_loss)
    s2_history["val_loss"].append(val_loss)
    s2_history["val_auc"].append(val_auc)
    s2_history["val_f1"].append(val_f1)
    scheduler_s2.step(val_loss)

    print(f"Ep {epoch:02d}/{EPOCHS_S2}  train={train_loss:.4f}  val={val_loss:.4f}  auc={val_auc:.4f}  f1={val_f1:.4f}")

    if val_loss < best_val_loss_s2:
        best_val_loss_s2, patience_counter_s2 = val_loss, 0
        torch.save(bag_model.state_dict(), S2_WEIGHTS_V2)
        print("  ✓ Saved best Stage 2 (v2) weights")
    else:
        patience_counter_s2 += 1
        if patience_counter_s2 >= PATIENCE_S2:
            print(f"  Early stopping at epoch {epoch}")
            break

print(f"\nStage 2 (v2) complete — {(time.time()-t0)/60:.1f} min")

Ep 01/30  train=0.1351  val=0.1019  auc=0.9869  f1=0.9680
  ✓ Saved best Stage 2 (v2) weights
Ep 02/30  train=0.0277  val=0.1019  auc=0.9853  f1=0.9716
  ✓ Saved best Stage 2 (v2) weights
Ep 03/30  train=0.0182  val=0.1039  auc=0.9868  f1=0.9716
Ep 04/30  train=0.0128  val=0.1085  auc=0.9870  f1=0.9789
Ep 05/30  train=0.0137  val=0.1110  auc=0.9871  f1=0.9789
Ep 06/30  train=0.0099  val=0.1095  auc=0.9865  f1=0.9789
Ep 07/30  train=0.0083  val=0.1092  auc=0.9857  f1=0.9753
Ep 08/30  train=0.0078  val=0.1094  auc=0.9857  f1=0.9787
Ep 09/30  train=0.0070  val=0.1095  auc=0.9859  f1=0.9823
  Early stopping at epoch 9

Stage 2 (v2) complete — 0.3 min


Cell 9 — ConvNeXt-Nano (v2) val/test evaluation, threshold calibration, save results

In [16]:
bag_model.load_state_dict(torch.load(S2_WEIGHTS_V2, map_location=DEVICE))
bag_model.eval()

val_bag_probs, val_bag_true = [], []
with torch.no_grad():
    for h_list, labels in val_bag_dl:
        h = h_list[0].to(DEVICE)
        logit, _ = bag_model(h)
        val_bag_probs.append(torch.sigmoid(logit).item())
        val_bag_true.append(labels[0].item())
val_bag_probs, val_bag_true = np.array(val_bag_probs), np.array(val_bag_true)

fpr_c, tpr_c, thresholds_c = roc_curve(val_bag_true, val_bag_probs)
idx = np.argmax(tpr_c >= 0.90)
optimal_threshold = float(thresholds_c[idx])
print(f"Calibrated threshold: {optimal_threshold:.4f}  sensitivity@cal: {tpr_c[idx]:.4f}  FPR@cal: {fpr_c[idx]:.4f}")

val_metrics_default = compute_all_metrics(val_bag_true, val_bag_probs, 0.5, "Val — default (0.5)")
val_metrics_cal = compute_all_metrics(val_bag_true, val_bag_probs, optimal_threshold, f"Val — calibrated ({optimal_threshold:.3f})")

test_bag_list = np.unique(bag_ids_test)
test_bag_ds = CachedBagDataset(feats_test_cnx, y_test_all, bag_ids_test, test_bag_list)
test_bag_dl = DataLoader(test_bag_ds, batch_size=1, shuffle=False, collate_fn=collate_cached)

test_bag_probs, test_bag_true = [], []
with torch.no_grad():
    for h_list, labels in test_bag_dl:
        h = h_list[0].to(DEVICE)
        logit, _ = bag_model(h)
        test_bag_probs.append(torch.sigmoid(logit).item())
        test_bag_true.append(labels[0].item())
test_bag_probs, test_bag_true = np.array(test_bag_probs), np.array(test_bag_true)

test_metrics = compute_all_metrics(test_bag_true, test_bag_probs, optimal_threshold, f"Test — calibrated ({optimal_threshold:.3f})")

results_v2 = {
    "model": "convnext_nano_v2_manual_fix",
    "feat_dim": FEAT_DIM,
    "attn_dim": ATTN_DIM,
    "stage1_metrics": s1_metrics_v2,
    "val_metrics_default": val_metrics_default,
    "val_metrics_cal": val_metrics_cal,
    "test_metrics": test_metrics,
    "optimal_threshold": optimal_threshold,
    "hyperparameters": {
        "lr_stage1": LR_STAGE1, "lr_s2_head": LR_S2_HEAD,
        "epochs_s1": EPOCHS_S1, "epochs_s2": EPOCHS_S2,
        "bs_stage1": BS_STAGE1, "attn_dim": ATTN_DIM,
        "patience_s1": PATIENCE_S1, "patience_s2": PATIENCE_S2,
    }
}
with open(RESULTS_JSON_V2, "w") as f:
    json.dump(results_v2, f, indent=2)
print("Saved:", RESULTS_JSON_V2)

print("\n── Comparison: ConvNeXt-Nano baseline (NB04) vs manual fix (v2) ──")
print(f"{'Metric':<20}{'Baseline (NB04)':>18}{'Manual fix (v2)':>18}")
baseline_test = {"auc": 0.9748, "f1": 0.9141, "sensitivity": 0.9048, "specificity": 0.9524, "fpr": 0.0476}
for k in ["auc", "f1", "sensitivity", "specificity", "fpr"]:
    print(f"{k:<20}{baseline_test[k]:>18.4f}{test_metrics[k]:>18.4f}")

Calibrated threshold: 0.8221  sensitivity@cal: 0.9444  FPR@cal: 0.0000
── Val — default (0.5) ──
  threshold      : 0.5000
  auc            : 0.9853
  f1             : 0.9716
  sensitivity    : 0.9514
  specificity    : 0.9912
  fpr            : 0.0088
  ece            : 0.0134
── Val — calibrated (0.822) ──
  threshold      : 0.8221
  auc            : 0.9853
  f1             : 0.9714
  sensitivity    : 0.9444
  specificity    : 1.0000
  fpr            : 0.0000
  ece            : 0.0134
── Test — calibrated (0.822) ──
  threshold      : 0.8221
  auc            : 0.9999
  f1             : 0.9898
  sensitivity    : 0.9864
  specificity    : 0.9957
  fpr            : 0.0043
  ece            : 0.0163
Saved: /kaggle/working/convnext_nano_v2_results.json

── Comparison: ConvNeXt-Nano baseline (NB04) vs manual fix (v2) ──
Metric                 Baseline (NB04)   Manual fix (v2)
auc                             0.9748            0.9999
f1                              0.9141            0.9898
se